<a href="https://colab.research.google.com/github/Himanshu90909/-Control-Structures-in-Python/blob/main/Excelerate_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("/content/Connect+Data+-+Connectapril2026+(1) (1).csv")
df

In [ ]:
df.shape

In [ ]:
print(f"Total number of rows: {df.shape[0]}")
print(f"Total number of columns: {df.shape[1]}")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include = 'object')

In [ ]:
a = df.isnull().sum()
a

In [ ]:
percentage = a / len(df) * 100
percentage

In [ ]:
missing_df = pd.DataFrame({'Missing Count': a, 'Missing Percentage (%)': percentage})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Percentage (%)', ascending=False)
print(missing_df)

In [ ]:
df1 = pd.read_csv("/content/SEVIS+Data+-+Sevisapril2026.csv")
df1

In [ ]:
df1.info()

In [ ]:
df1.describe()

In [ ]:
df1.describe(include = 'all')

In [ ]:
a1 = df.isnull().sum()
a1

In [ ]:
percentage1 = a1 / len(df1) * 100
percentage1

In [ ]:
missing_df1 = pd.DataFrame({'Missing Count': a1, 'Missing Percentage (%)': percentage1})
missing_df1 = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Percentage (%)', ascending=False)
print(missing_df1)

In [ ]:
df2 = pd.read_csv("/content/Copy of Connect - SLUConnect.csv")
df2

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include = 'all')

In [ ]:
a2 = df.isnull().sum()
a2

In [ ]:
percentage2 = a2 / len(df2) * 100
percentage2

In [ ]:
missing_df2 = pd.DataFrame({'Missing Count': a2, 'Missing Percentage (%)': percentage2})
missing_df2 = missing_df2[missing_df2['Missing Count'] > 0].sort_values(by='Missing Percentage (%)', ascending=False)
print(missing_df2)

In [ ]:
df_connect = pd.read_csv("/content/Connect+Data+-+Connectapril2026+(1) (1).csv")
df_sevis = pd.read_csv("/content/SEVIS+Data+-+Sevisapril2026.csv")
df_connect_copy = pd.read_csv("/content/Copy of Connect - SLUConnect.csv")

In [ ]:
df_sevis['NonImmigrant_ID_cleaned'] = pd.to_numeric(df_sevis['NonImmigrant_ID'], errors='coerce')
df_connect['Reference_ID'] = pd.to_numeric(df_connect['Reference_ID'], errors='coerce')

In [ ]:
print("--- Joining Connect and SEVIS Data ---")
print(f"Connect Dataset (df_connect) initial rows: {len(df_connect)}")
print(f"SEVIS Dataset (df_sevis) initial rows: {len(df_sevis)}")

In [ ]:
merged_connect_sevis = pd.merge(
    df_connect,
    df_sevis,
    left_on='Reference_ID',
    right_on='NonImmigrant_ID_cleaned',
    how='left',
    suffixes=('_connect', '_sevis')
)

In [ ]:

print(f"Merged Connect and SEVIS Dataset rows (left join): {len(merged_connect_sevis)}")
print("\nFirst 5 rows of merged_connect_sevis:")
display(merged_connect_sevis.head())

In [ ]:

# Record reconciliation checks: Check how many records from Connect found a match in SEVIS
matched_records_count = merged_connect_sevis['SEVIS_ID'].notna().sum()
print(f"Number of Connect records that found a match in SEVIS: {matched_records_count}")
print(f"Number of Connect records that did NOT find a match in SEVIS: {len(df_connect) - matched_records_count}")

In [ ]:

unmatched_connect = merged_connect_sevis[merged_connect_sevis['SEVIS_ID'].isna()]
print(f"Percentage of Connect records without a SEVIS match: {len(unmatched_connect) / len(df_connect) * 100:.2f}%")

In [ ]:

print("\n--- Joining Original Connect and Copy Connect Data ---")
print(f"Original Connect Dataset (df_connect) initial rows: {len(df_connect)}")
print(f"Copy Connect Dataset (df_connect_copy) initial rows: {len(df_connect_copy)}")

In [ ]:
merged_connect_comparison = pd.merge(
    df_connect,
    df_connect_copy,
    on='Reference_ID',
    how='inner',
    suffixes=('_original', '_copy')
)

print(f"Merged Original Connect and Copy Connect Dataset rows (inner join): {len(merged_connect_comparison)}")
print("\nFirst 5 rows of merged_connect_comparison:")
display(merged_connect_comparison.head())

# For comprehensive reconciliation, a full outer join would identify discrepancies.
full_merged_connect_comparison = pd.merge(
    df_connect,
    df_connect_copy,
    on='Reference_ID',
    how='outer',
    suffixes=('_original', '_copy'),
    indicator=True
)

# Records only in original Connect
only_in_original = full_merged_connect_comparison[full_merged_connect_comparison['_merge'] == 'left_only']
# Records only in copy Connect
only_in_copy = full_merged_connect_comparison[full_merged_connect_comparison['_merge'] == 'right_only']

print(f"Records present only in Original Connect: {len(only_in_original)}")
print(f"Records present only in Copy Connect: {len(only_in_copy)}")

if not only_in_original.empty:
    print("Example records present only in Original Connect:")
    display(only_in_original.head())
if not only_in_copy.empty:
    print("Example records present only in Copy Connect:")
    display(only_in_copy.head())

In [ ]:
print("\n**Data Limitations and Risks:**")
print("1. **Connect and SEVIS Integration:**")
print("   - The join between Connect and SEVIS datasets was performed using 'Reference_ID' and 'NonImmigrant_ID_cleaned'.")
print(f"   - Out of {len(df_connect)} Connect records, {matched_records_count} found a match in SEVIS. This means {len(df_connect) - matched_records_count} Connect records ({len(unmatched_connect) / len(df_connect) * 100:.2f}%) did not have a corresponding entry in the SEVIS dataset based on the chosen key.")
print("   - This high number of unmatched records could indicate a low overlap between the two systems using 'NonImmigrant_ID', or issues with the key itself (e.g., SEVIS 'NonImmigrant_ID' often contained '#FMT' or was missing for many SEVIS records).")
print("   - The original 'NonImmigrant_ID' in SEVIS contained non-numeric values ('#FMT'), which required cleaning (coercing to NaN) before joining. This could also contribute to unmatched records.")
print("\n2. **Connect (Original) and Connect (Copy) Integration:**")
print("   - Both 'Connect' datasets were joined on 'Reference_ID' to check for consistency.")
print(f"   - The inner join resulted in {len(merged_connect_comparison)} rows, matching the original dataset sizes. This suggests the two 'Connect' datasets are identical in terms of 'Reference_ID' coverage.")
print(f"   - Explicit check with outer join showed {len(only_in_original)} records only in original and {len(only_in_copy)} records only in copy, indicating a high degree of similarity in terms of reference IDs.")
print("\n**Identified Inconsistencies/Limitations:**")
print("   - The primary challenge for integration is the low match rate between Connect and SEVIS using 'NonImmigrant_ID'. Further investigation is needed to confirm the correct linking key between these two systems.")
print("   - The presence of `#FMT` in `SEVIS['NonImmigrant_ID']` required cleaning, which might discard useful information if `#FMT` has specific meaning or can be mapped.")
print("   - Data quality in `FIN_ID` (mostly missing) in SEVIS and other heavily missing columns across all datasets will affect the richness of the integrated dataset.")